# SPARK ML

- Spark에서 머신 러닝 및 데이터 마이닝 작업을 수행하기 위한 라이브러리.

## 1. Linear Regression

In [1]:
!pip3 install pyspark

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.0/317.0 MB 3.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pyspark: filename=pyspark-3.5.1-py2.py3-none-any.whl size=317488491 sha256=4dbabed098da9a228fe11dab0caa3d8cb72edffe286fd619afa6aea2d8a1b660
  Stored in directory: /root/.cache/pip/wheels/80/1d/60/2c256ed38dddce2fdd93be545214a63e02fbd8d74fb0b7f3a6
Successfully built pyspark


In [7]:
from pyspark.sql import SparkSession
from pyspark.sql import types as T
from pyspark.sql import  functions as F
spark = SparkSession.builder.master("local").appName("colab").getOrCreate()

In [22]:
house = spark.read.csv("/content/sample_data/california_housing_train.csv",header=True)
house.show(3)

+-----------+---------+------------------+-----------+--------------+-----------+----------+-------------+------------------+
|  longitude| latitude|housing_median_age|total_rooms|total_bedrooms| population|households|median_income|median_house_value|
+-----------+---------+------------------+-----------+--------------+-----------+----------+-------------+------------------+
|-114.310000|34.190000|         15.000000|5612.000000|   1283.000000|1015.000000|472.000000|     1.493600|      66900.000000|
|-114.470000|34.400000|         19.000000|7650.000000|   1901.000000|1129.000000|463.000000|     1.820000|      80100.000000|
|-114.560000|33.690000|         17.000000| 720.000000|    174.000000| 333.000000|117.000000|     1.650900|      85700.000000|
+-----------+---------+------------------+-----------+--------------+-----------+----------+-------------+------------------+
only showing top 3 rows



In [23]:
for col in house.columns:
  house = house.withColumn(col, F.col(col).cast(T.FloatType()))

In [24]:
house.dtypes

[('longitude', 'float'),
 ('latitude', 'float'),
 ('housing_median_age', 'float'),
 ('total_rooms', 'float'),
 ('total_bedrooms', 'float'),
 ('population', 'float'),
 ('households', 'float'),
 ('median_income', 'float'),
 ('median_house_value', 'float')]

In [25]:
#house테이블 형상 출력

print("Shape : " , (house.count(),len(house.columns)))

Shape :  (17000, 9)


In [26]:
from pyspark.ml.feature import VectorAssembler
featureAssembler=VectorAssembler(inputCols=house.columns[:-1],outputCol="features")

In [27]:
output = featureAssembler.transform(house)

In [28]:
# featureAssembler.transform(house).show(1, vertical=True, truncate=False)

In [29]:
house_data = output.select('features', 'median_house_value')
house_data.show(4)

+--------------------+------------------+
|            features|median_house_value|
+--------------------+------------------+
|[-114.30999755859...|           66900.0|
|[-114.47000122070...|           80100.0|
|[-114.55999755859...|           85700.0|
|[-114.56999969482...|           73400.0|
+--------------------+------------------+
only showing top 4 rows



In [31]:
train_data, test_data=house_data.randomSplit(weights=[0.8,0.2], seed=42)

In [32]:
(train_data.count(), len(train_data.columns)), (test_data.count(), len(test_data.columns))

((13671, 2), (3329, 2))

In [33]:
from pyspark.ml.regression import LinearRegression
lr_model=LinearRegression(featuresCol="features", labelCol="median_house_value")

In [34]:
lr_fit=lr_model.fit(train_data)

In [36]:
train_data.show(1, vertical=True, truncate=False)

-RECORD 0-----------------------------------------------------------------------------------------------------
 features           | [-124.3499984741211,40.540000915527344,52.0,1820.0,300.0,806.0,270.0,3.014699935913086] 
 median_house_value | 94600.0                                                                                 
only showing top 1 row



In [35]:
lr_fit.coefficients

DenseVector([-42897.9984, -42705.3007, 1144.0811, -8.2234, 119.8248, -37.871, 40.8358, 40399.1481])

In [39]:
pred = lr_fit.evaluate(test_data)
pred.predictions.show()

+--------------------+------------------+------------------+
|            features|median_house_value|        prediction|
+--------------------+------------------+------------------+
|[-124.30000305175...|          103600.0|101334.27515698876|
|[-124.23000335693...|          106700.0|189023.24584240932|
|[-124.23000335693...|           73200.0| 76470.00500450889|
|[-124.19000244140...|           90100.0|165186.21483835625|
|[-124.18000030517...|           67000.0| 120101.4706942197|
|[-124.16999816894...|          116100.0|199313.13389420137|
|[-124.16999816894...|           62500.0|132221.09111186163|
|[-124.16000366210...|           85400.0|157664.88765429985|
|[-124.15000152587...|           90000.0|174691.87089904258|
|[-124.15000152587...|           86400.0| 157459.5013977755|
|[-124.15000152587...|           74100.0| 121336.4909311342|
|[-124.15000152587...|           57500.0| 104446.8328753165|
|[-124.13999938964...|           75100.0|134649.33405189076|
|[-124.13999938964...|  

In [41]:
print("R2 score for LR model :",pred.r2)

R2 score for LR model : 0.6420168570584976






## 2. Random Forest

In [69]:
iris = spark.read.csv("./iris.csv",header=True)
iris.show(3)

+------------+-----------+------------+-----------+-------+
|sepal_length|sepal_width|petal_length|petal_width|variety|
+------------+-----------+------------+-----------+-------+
|         5.1|        3.5|         1.4|         .2| Setosa|
|         4.9|          3|         1.4|         .2| Setosa|
|         4.7|        3.2|         1.3|         .2| Setosa|
+------------+-----------+------------+-----------+-------+
only showing top 3 rows



In [46]:
iris.printSchema()

root
 |-- sepal_length: string (nullable = true)
 |-- sepal_width: string (nullable = true)
 |-- petal_length: string (nullable = true)
 |-- petal_width: string (nullable = true)
 |-- variety: string (nullable = true)



In [71]:
for col in iris.columns[:-1]:
  iris= iris.withColumn(col,F.col(col).cast(T.FloatType()))
iris.printSchema()
# for col in house.columns:
#   house = house.withColumn(col, F.col(col).cast(T.FloatType()))

root
 |-- sepal_length: float (nullable = true)
 |-- sepal_width: float (nullable = true)
 |-- petal_length: float (nullable = true)
 |-- petal_width: float (nullable = true)
 |-- variety: string (nullable = true)



In [72]:
from pyspark.ml.feature import VectorAssembler
featureAssembler = VectorAssembler(inputCols=iris.columns[:-1],outputCol='features')

In [73]:
iris_data = featureAssembler.transform(iris)
iris_data = iris_data.select("features","variety")

iris_data.show(5)

+--------------------+-------+
|            features|variety|
+--------------------+-------+
|[5.09999990463256...| Setosa|
|[4.90000009536743...| Setosa|
|[4.69999980926513...| Setosa|
|[4.59999990463256...| Setosa|
|[5.0,3.5999999046...| Setosa|
+--------------------+-------+
only showing top 5 rows



In [74]:
from pyspark.ml.feature import StringIndexer
String_indexer=StringIndexer(inputCol="variety", outputCol="variety_index")
iris_indexed = String_indexer.fit(iris_data).transform(iris_data)
iris_indexed.select("variety","variety_index").distinct().show()

+----------+-------------+
|   variety|variety_index|
+----------+-------------+
| Virginica|          2.0|
|Versicolor|          1.0|
|    Setosa|          0.0|
+----------+-------------+



In [75]:
(iris_indexed.count(), len(iris_indexed.columns))

(150, 3)

In [76]:
train_data, test_data = iris_indexed.randomSplit([0.7,0.3], seed=42)
(train_data.count(),len(train_data.columns),(test_data.count()),len(test_data.columns))

(104, 3, 46, 3)

In [77]:
from pyspark.ml.classification import RandomForestClassifier
rf_model=RandomForestClassifier(featuresCol="features",labelCol="variety_index")

In [78]:
rf_train = rf_model.fit(train_data)

In [79]:
pred=rf_train.transform(test_data)

pred.show()

+--------------------+----------+-------------+--------------------+--------------------+----------+
|            features|   variety|variety_index|       rawPrediction|         probability|prediction|
+--------------------+----------+-------------+--------------------+--------------------+----------+
|[4.40000009536743...|    Setosa|          0.0|      [20.0,0.0,0.0]|       [1.0,0.0,0.0]|       0.0|
|[4.59999990463256...|    Setosa|          0.0|      [20.0,0.0,0.0]|       [1.0,0.0,0.0]|       0.0|
|[4.59999990463256...|    Setosa|          0.0|      [20.0,0.0,0.0]|       [1.0,0.0,0.0]|       0.0|
|[4.69999980926513...|    Setosa|          0.0|      [20.0,0.0,0.0]|       [1.0,0.0,0.0]|       0.0|
|[4.80000019073486...|    Setosa|          0.0|      [20.0,0.0,0.0]|       [1.0,0.0,0.0]|       0.0|
|[4.80000019073486...|    Setosa|          0.0|      [20.0,0.0,0.0]|       [1.0,0.0,0.0]|       0.0|
|[4.80000019073486...|    Setosa|          0.0|      [15.0,5.0,0.0]|     [0.75,0.25,0.0]|  

In [80]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

evaluator = MulticlassClassificationEvaluator(labelCol="variety_index", predictionCol="prediction")
accuracy = evaluator.evaluate(pred)
print("Accuracy = %s" % (accuracy))
print("Test Error = %s" % (1.0 - accuracy))

Accuracy = 0.978458139351377
Test Error = 0.021541860648622957


In [81]:
from pyspark.mllib.evaluation import MulticlassMetrics
from pyspark.sql import types as T
from pyspark.sql import functions as F

preds_and_labels = pred.select(['prediction','variety_index']).withColumn('variety_index', F.col('variety_index').cast(T.FloatType())).orderBy('prediction')
preds_and_labels = preds_and_labels.select(['prediction','variety_index'])
metrics = MulticlassMetrics(preds_and_labels.rdd.map(tuple))
print(metrics.confusionMatrix().toArray())

/usr/local/lib/python3.10/dist-packages/pyspark/sql/context.py:158: FutureWarning: Deprecated in 3.0.0. Use SparkSession.builder.getOrCreate() instead.
  warnings.warn(


[[22.  0.  0.]
 [ 0. 14.  1.]
 [ 0.  0.  9.]]
